# 01 — Baseline: ordinary GPT, fixed batch size

Trains the standard (non-reversible) 20M-parameter GPT for 50M tokens of TinyStories at a **fixed batch size of 32 × 256 tokens**.
This is the reference point every other run is compared against. Expected time on a Colab T4: ~25–40 min.

Results are written to `results/baseline_B32.json` (loss curve, tokens/s, peak memory, config).

In [ ]:
# @title Setup — clone repo (if needed), install deps, detect GPU
import os, sys, subprocess, json, time, math
REPO_URL = "https://github.com/swatibansal/reversible-llm-poc.git"

if not os.path.exists("src/revllm.py"):
    if os.path.exists("../src/revllm.py"):
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, "reversible-llm-poc"], check=True)
        os.chdir("reversible-llm-poc")
sys.path.insert(0, os.path.abspath("src"))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tiktoken", "datasets", "matplotlib"], check=False)

import torch
from revllm import Config, GPT, SavedTensorMeter
from data import prepare_tinystories, prepare_synthetic, TokenStream
import train as T

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# SMOKE mode = tiny synthetic run that finishes in ~1 min on CPU. Auto-enabled when there is no GPU.
SMOKE = os.environ.get("SMOKE", "0") == "1" or DEVICE == "cpu"
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu", "| SMOKE mode:", SMOKE)
os.makedirs("results", exist_ok=True)


In [ ]:
# @title Experiment configuration (shared by all notebooks)
if SMOKE:
    MODEL  = dict(vocab_size=512, block_size=64, n_layer=4, n_embd=128, n_head=4)
    TOKENS = 200_000          # token budget
    BATCH  = 16               # the fixed batch size for notebooks 01/02
    LR     = 2e-3
    LOG    = dict(eval_every=100, eval_iters=5, log_every=50)
else:
    # ~20.8M parameters (12.9M in the tied GPT-2 embedding, 7.9M in 10 transformer blocks of width 256)
    MODEL  = dict(vocab_size=50257, block_size=256, n_layer=10, n_embd=256, n_head=4)
    TOKENS = 50_000_000
    BATCH  = 32               # 32 x 256 = 8,192 tokens / step  -> ~6,100 steps for 50M tokens
    LR     = 6e-4
    LOG    = dict(eval_every=250, eval_iters=20, log_every=50)

def get_data():
    if SMOKE:
        return prepare_synthetic("data/synthetic", 2_000_000, 200_000, vocab=MODEL["vocab_size"])
    return prepare_tinystories("data/tinystories", n_train_tokens=55_000_000, n_val_tokens=2_000_000)

def gpu_table(rows, headers):
    w = [max(len(str(r[i])) for r in [headers] + rows) for i in range(len(headers))]
    line = lambda r: "| " + " | ".join(str(c).ljust(w[i]) for i, c in enumerate(r)) + " |"
    print(line(headers)); print("|" + "|".join("-" * (x + 2) for x in w) + "|")
    for r in rows: print(line(r))

import matplotlib.pyplot as plt
def plot_runs(results, key="curve", title="training loss", smooth=25):
    plt.figure(figsize=(8, 4.5))
    for r in results:
        c = r[key]
        if not c: continue
        xs = [p[1] / 1e6 for p in c]; ys = [p[2] for p in c]
        if key == "curve" and smooth > 1 and len(ys) > smooth:
            ys = [sum(ys[max(0, i - smooth):i + 1]) / len(ys[max(0, i - smooth):i + 1]) for i in range(len(ys))]
        plt.plot(xs, ys, label=f"{r['run_name']}  (final {ys[-1]:.3f})")
    plt.xlabel("tokens seen (M)"); plt.ylabel("cross-entropy (nats)"); plt.title(title); plt.legend(); plt.grid(alpha=.3)
    plt.show()


In [ ]:
train_bin, val_bin = get_data()
cfg = Config(mode="baseline", **MODEL)
res, model = T.train(cfg, train_bin, val_bin, batch_size=BATCH, tokens_budget=TOKENS, lr=LR,
                     out_json=f"results/baseline_B{BATCH}.json", **LOG)

In [ ]:
plot_runs([res], title="baseline — training loss")
plot_runs([res], key="val_curve", title="baseline — validation loss")
print(json.dumps({k: v for k, v in res.items() if k not in ("curve", "val_curve", "config")}, indent=1))

## Sample generation (sanity check that it learned *something*)

In [ ]:
if not SMOKE:
    import tiktoken; enc = tiktoken.get_encoding("gpt2")
    prompt = torch.tensor([enc.encode("Once upon a time")], device=DEVICE)
    model.eval()
    with torch.no_grad(), T._amp(DEVICE)[0]:
        out = model.generate(prompt, 80, temperature=0.8, top_k=50)
    print(enc.decode(out[0].tolist()))